In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # COLEPV1
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pprint import pprint

from colep_ai.database.azure_search_client import get_search_client
from colep_ai.indexing.embedder import get_openai_client
from colep_ai.retrieval.ai_search_retrieval import (
    retrieve,
    RetrievalResponse,
    RetrievalRejected,
)

In [3]:
openai_client = get_openai_client()
search_client = get_search_client()

2026-07-30 18:10:40 | INFO | colep_ai.database.azure_search_client | Search client initiated |index =colep-page-based-chunks_new_emb | 



In [ ]:
query = " For the purposes of the inspection plan, which situations are considered a “start-up”? "
result = await retrieve(
    query=query,
    openai_client=openai_client,
    search_client=search_client, 
    top_k=20,
)

2026-07-30 18:10:40 | INFO | colep_ai.retrieval.ai_search_retrieval | Language detected: 'en' (1.00) | query=' For the purposes of the inspection plan, which situations a'



2026-07-30 18:10:43 | INFO | colep_ai.retrieval.ai_search_retrieval | Embedding generation took 2.296s

2026-07-30 18:10:44 | INFO | colep_ai.retrieval.ai_search_retrieval | Azure search returned 20 results above threshold | field=(text_en,vector_text_en) | filter=None

2026-07-30 18:10:44 | INFO | colep_ai.retrieval.ai_search_retrieval | Retrieval took 1.630s

2026-07-30 18:10:44 | INFO | colep_ai.retrieval.ai_search_retrieval | Retrieval complete | language=en | line_filter=None | results=20 | top_score=0.033333



In [5]:
print(type(result))


<class 'colep_ai.retrieval.ai_search_retrieval.RetrievalResponse'>


In [6]:
if isinstance(result, RetrievalRejected):
    print(result.reason)

elif isinstance(result, RetrievalResponse):
    print("Language:", result.language)
    print("Line Filter:", result.line_filter)
    print("Results:", len(result.results))

Language: en
Line Filter: None
Results: 20


In [7]:
for i, r in enumerate(result.results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print(f"Score         : {r['score']:.3f}")
    print(f"Source File   : {r['source_file']}")
    # print(f"folder_name   : {r['folder_name']}")
    print(f"Document Code : {r['document_code']}")
    print(f"Title         : {r['document_title']}")
    print(f"Page          : {r['page_number']}")
    print(f"Line Numbers  : {r['line_number']}")
    print(f"page_image_ids  : {r['page_image_ids']}")

    # print("\nflowchart")
    # pprint(r["flowchart"])

Result 1
Score         : 0.033
Source File   : Q01_O048_1_Controlo_da_porosidade_Estampagem_P_C
Document Code : Q01.O048.1
Title         : Controlo da porosidade - Q01.O048.1
Page          : 2
Line Numbers  : []
page_image_ids  : []
Result 2
Score         : 0.033
Source File   : Q01_O048_1_Controlo_da_porosidade_Estampagem_P_C
Document Code : 
Title         : Operação nº01 – Controlo da porosidade
Page          : 4
Line Numbers  : []
page_image_ids  : []
Result 3
Score         : 0.031
Source File   : Q01_E018_2_Ficha_de_Especificacao_CY_153
Document Code : Q01.E018.2
Title         : Especificações Limites CY 153
Page          : 1
Line Numbers  : []
page_image_ids  : []
Result 4
Score         : 0.031
Source File   : Q01_T016_1_Paletes_alternativas
Document Code : 
Title         : Tabela de Parâmetros por Formato
Page          : 4
Line Numbers  : []
page_image_ids  : []
Result 5
Score         : 0.031
Source File   : Q01_M156_2_Mapa_de_controlo_da_Estampagem_Linhas_28_34_e_95
Document Cod

In [8]:

from colep_ai.retrieval.ai_search_retrieval import retrieve, RetrievalRejected,RetrievalResponse
from colep_ai.retrieval.reranker import rerank


In [9]:
 # ------------------------------------------------------------------
    # 5b. Rerank
# ------------------------------------------------------------------

retrieval_response = RetrievalResponse(
    results=await rerank(query, result.results),
    language=result.language,
    line_filter=result.line_filter,
)


2026-07-30 18:12:59 | INFO | colep_ai.retrieval.reranker | Reranked 20 → 10 | top_score=0.063948



In [10]:
retrieval_response

RetrievalResponse(results=[{'id': 'ea8d2fe3-0b18-56de-a46d-2d36f90b6f5b', 'source_file': 'Q01_O048_1_Controlo_da_porosidade_Estampagem_P_C', 'document_code': 'Q01.O048.1', 'document_title': 'Controlo da porosidade - Q01.O048.1', 'page_number': 2, 'line_number': [], 'page_image_ids': [], 'text_pt': 'Controlo da porosidade - Q01.O048.1 No arranque e/ou de acordo com o Plano de Inspeção e Ensaio, retirar amostras do final de linha. Colocar os componentes numa solução de sulfato de cobre a 25%, durante 30 segundos. Remover o sulfato de cobre, mergulhando os componentes em água. Observar visualmente os componentes e registar no mapa Q01.M001. Na fase 1 e 4 é obrigatório o uso de luvas de proteção.\nNas fases 2 e 3 é obrigatório o uso de luvas de proteção química.', 'text_en': 'Controlo da porosidade - Q01.O048.1 At startup and/or in accordance with the Inspection and Testing Plan, take samples from the end of the line. Place the components in a 25% copper sulfate solution for 30 seconds. Re

In [11]:
for i, r in enumerate(retrieval_response.results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print(f"Score         : {r['score']:.3f}")
    print(f"Source File   : {r['source_file']}")
    # print(f"folder_name   : {r['folder_name']}")
    print(f"Document Code : {r['document_code']}")
    print(f"Title         : {r['document_title']}")
    print(f"Page          : {r['page_number']}")
    print(f"Line Numbers  : {r['line_number']}")
    print(f"page_image_ids  : {r['page_image_ids']}")


Result 1
Score         : 0.033
Source File   : Q01_O048_1_Controlo_da_porosidade_Estampagem_P_C
Document Code : Q01.O048.1
Title         : Controlo da porosidade - Q01.O048.1
Page          : 2
Line Numbers  : []
page_image_ids  : []
Result 2
Score         : 0.033
Source File   : Q01_O048_1_Controlo_da_porosidade_Estampagem_P_C
Document Code : 
Title         : Operação nº01 – Controlo da porosidade
Page          : 4
Line Numbers  : []
page_image_ids  : []
Result 3
Score         : 0.015
Source File   : Q01_L132_3_Plano_de_Inspecao_e_Ensaio_L19_e_83_Estampagem
Document Code : Q01.M042.1
Title         : Plano de Inspeção e Ensaio - Q01.L132.3
Page          : 1
Line Numbers  : [19, 83]
page_image_ids  : []
Result 4
Score         : 0.015
Source File   : Q01_M156_2_Mapa_de_controlo_da_Estampagem_Linhas_28_34_e_95
Document Code : Q01.M156.2/2
Title         : MAPA DE REGISTO DE CONTROLO E PRODUÇÃO DE ESTAMPAGEM
Page          : 2
Line Numbers  : [28, 34, 95]
page_image_ids  : []
Result 5
Score  

In [12]:
from colep_ai.generation.ai_search_generation_session import generate_from_retrieval
from colep_ai.generation.claude_client import get_claude_client

In [13]:
claude_client=get_claude_client()

In [14]:
history_messages=[]

In [15]:
generation_output = await generate_from_retrieval(
        claude_client=claude_client,
        query=query,
        retrieval_response=retrieval_response,
        history_messages=history_messages,
    )

2026-07-30 18:16:58 | INFO | colep_ai.generation.ai_search_generation_session | Generating answer | model=claude-sonnet-4-6 | answer_language=English | history_turns=0

2026-07-30 18:17:04 | INFO | colep_ai.generation.ai_search_generation_session | Token usage | input=24644 | output=121 | total=24765

2026-07-30 18:17:04 | INFO | colep_ai.generation.ai_search_generation_session | RAW ANSWER:
'Based on the inspection plan, the following situations all count as a "start-up":\n\n1. Beginning a new Manufacturing Order.\n2. Resuming a Manufacturing Order that was previously interrupted.\n3. Changing shifts.\n4. Any stoppage lasting more than 4 hours.\n\n📄 Source\n- File: Q01_L132_3_Plano_de_Inspecao_e_Ensaio_L19_e_83_Estampagem\n- Line Number: 19, 83'



In [19]:
generation_output["answer"]

'Based on the inspection plan, the following situations all count as a "start-up":\n\n1. Beginning a new Manufacturing Order.\n2. Resuming a Manufacturing Order that was previously interrupted.\n3. Changing shifts.\n4. Any stoppage lasting more than 4 hours.\n\n📄 Source\n- File: Q01_L132_3_Plano_de_Inspecao_e_Ensaio_L19_e_83_Estampagem\n- Line Number: 19, 83'